# Phase 1: Data Infrastructure & Empirical Diagnostics  
## Notebook 01_08 — Data Leakage Detection Unit Tests

### Research question

How can we build automated tests that detect look-ahead bias, target leakage, preprocessing leakage, random-split leakage, and overlapping-label leakage before a financial machine learning or backtesting notebook reports performance?

This notebook builds a small testing framework for financial time-series research.

It deliberately creates:

1. a **safe feature panel**;
2. a **leaky feature panel**;
3. a **time-ordered train/test split**;
4. a **random split that violates time order**;
5. **overlapping labels**;
6. **purged and embargoed splits**;
7. **metadata-based feature availability checks**;
8. **preprocessing leakage checks**;
9. **pytest-style unit tests** that can be moved into the repository's `tests/` folder.

The goal is to make fake alpha fail loudly.

A useful principle:

> If a signal uses information that was unavailable at prediction time, it is not alpha. It is leakage.

## 1. Why leakage is dangerous in finance

Data leakage happens when information from the future, the test set, or the target variable enters the training process.

In ordinary machine learning, leakage is bad.

In finance, it is catastrophic because signals are weak, noisy, and expensive to validate. Even tiny leakage can create a beautiful backtest that completely disappears live.

Common leakage types include:

| Leakage type | Example |
|---|---|
| Look-ahead feature leakage | A feature at date $t$ uses returns from $t+1$ |
| Target leakage | A feature is the future return label itself |
| Centered rolling-window leakage | A rolling average uses both past and future observations |
| Global preprocessing leakage | A scaler is fitted using train and test data together |
| Random split leakage | A time series is randomly shuffled before train/test splitting |
| Overlapping-label leakage | Training labels overlap in time with test labels |
| Survivorship leakage | The universe contains only assets that survived to the end |
| Availability-time leakage | A fundamental/macro field is used before it was released |

This notebook focuses on the first six, while also introducing the availability-time logic needed for fundamentals and corporate-action data.

## 2. Labelling convention

Suppose a feature is observed at time $t$.

A forward return label over horizon $h$ is:

$$
y_t = \frac{P_{t+h}}{P_t} - 1
$$
or in log-return form:

$$
y_t = \log P_{t+h} - \log P_t
$$
The label interval is:

$$
(t, t+h]
$$
For the feature to be safe, all source data used to compute the feature must be available at or before $t$.

If the feature uses any information from $t+1$ to $t+h$, it leaks into the label.

## 3. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

In [ ]:
@dataclass(frozen=True)
class LeakageTestConfig:
    """
    Configuration for the synthetic leakage test environment.
    """
    start: str = "2020-01-01"
    periods: int = 1_250
    num_assets: int = 40
    label_horizon: int = 5
    train_fraction: float = 0.70
    embargo_days: int = 5
    seed: int = 42


config = LeakageTestConfig()
config

## 4. Synthetic return panel

We simulate a multi-asset return panel with weak autocorrelation and a common market factor.

This gives the safe features a small but realistic amount of predictive structure.

We then create leaky features that are obviously too good.

In [ ]:
def simulate_return_panel(config: LeakageTestConfig) -> pd.DataFrame:
    """
    Simulate a synthetic multi-asset return panel.
    """
    local_rng = np.random.default_rng(config.seed)

    dates = pd.date_range(
        config.start,
        periods=config.periods,
        freq="B",
        tz="UTC"
    )

    symbols = [f"Asset_{i:03d}" for i in range(1, config.num_assets + 1)]

    market = local_rng.normal(loc=0.0002, scale=0.010, size=config.periods)

    frames = []

    for symbol in symbols:
        beta = local_rng.normal(loc=1.0, scale=0.25)
        idio_vol = local_rng.uniform(0.008, 0.020)

        returns = np.zeros(config.periods)
        noise = local_rng.normal(loc=0.0, scale=idio_vol, size=config.periods)

        # Weak autocorrelation and market exposure.
        for t in range(1, config.periods):
            returns[t] = (
                0.05 * returns[t - 1]
                + beta * market[t]
                + noise[t]
            )

        prices = 100 * np.exp(np.cumsum(returns))

        frames.append(pd.DataFrame({
            "timestamp": dates,
            "symbol": symbol,
            "return": returns,
            "price": prices,
            "market_return": market,
            "beta": beta
        }))

    panel = pd.concat(frames, ignore_index=True)
    panel = panel.sort_values(["symbol", "timestamp"]).reset_index(drop=True)

    return panel


panel = simulate_return_panel(config)

panel.head()

## 5. Building labels

The label is the future log return over the next $h$ business days:

$$
y_t = \log P_{t+h} - \log P_t
$$
The label starts after the feature timestamp and ends at $t+h$.

We store:

- `feature_timestamp`;
- `label_start_time`;
- `label_end_time`.

These metadata columns are essential for purging and embargo tests.

In [ ]:
def add_forward_return_labels(
    panel: pd.DataFrame,
    horizon: int
) -> pd.DataFrame:
    """
    Add forward log-return labels and label interval metadata.
    """
    out = panel.sort_values(["symbol", "timestamp"]).copy()

    out["log_price"] = np.log(out["price"])
    out["future_log_price"] = out.groupby("symbol")["log_price"].shift(-horizon)
    out["label_forward_log_return"] = out["future_log_price"] - out["log_price"]

    out["feature_timestamp"] = out["timestamp"]
    out["label_start_time"] = out.groupby("symbol")["timestamp"].shift(-1)
    out["label_end_time"] = out.groupby("symbol")["timestamp"].shift(-horizon)

    out = out.dropna(subset=[
        "label_forward_log_return",
        "label_start_time",
        "label_end_time"
    ]).copy()

    return out


labelled_panel = add_forward_return_labels(panel, horizon=config.label_horizon)

labelled_panel[[
    "timestamp",
    "symbol",
    "return",
    "label_forward_log_return",
    "label_start_time",
    "label_end_time"
]].head()

## 6. Safe feature engineering

Safe features only use information available at or before the feature timestamp.

Examples:

1. previous day's return;
2. trailing 5-day return;
3. trailing 21-day volatility;
4. trailing market return;
5. cross-sectional rank computed using same-day observable returns.

The key point is that rolling windows must be backward-looking, not centered.

In [ ]:
def build_safe_features(labelled: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build safe backward-looking features and a feature catalog.
    """
    out = labelled.sort_values(["symbol", "timestamp"]).copy()

    grouped = out.groupby("symbol")

    out["feature_lag1_return"] = grouped["return"].shift(1)

    out["feature_trailing_5d_return"] = (
        grouped["return"]
        .rolling(window=5, min_periods=5)
        .sum()
        .reset_index(level=0, drop=True)
        .shift(1)
    )

    out["feature_trailing_21d_vol"] = (
        grouped["return"]
        .rolling(window=21, min_periods=21)
        .std()
        .reset_index(level=0, drop=True)
        .shift(1)
    )

    out["feature_trailing_market_5d"] = (
        out.groupby("symbol")["market_return"]
        .rolling(window=5, min_periods=5)
        .sum()
        .reset_index(level=0, drop=True)
        .shift(1)
    )

    # Cross-sectional same-day rank. This can be safe only if the strategy trades after the close.
    out["feature_same_day_cross_sectional_rank"] = (
        out.groupby("timestamp")["return"]
        .rank(pct=True)
    )

    feature_catalog = pd.DataFrame([
        {
            "feature": "feature_lag1_return",
            "earliest_source_offset": -1,
            "latest_source_offset": -1,
            "available_offset": 0,
            "description": "Previous day's return."
        },
        {
            "feature": "feature_trailing_5d_return",
            "earliest_source_offset": -5,
            "latest_source_offset": -1,
            "available_offset": 0,
            "description": "Trailing 5-day return ending before prediction date."
        },
        {
            "feature": "feature_trailing_21d_vol",
            "earliest_source_offset": -21,
            "latest_source_offset": -1,
            "available_offset": 0,
            "description": "Trailing 21-day volatility ending before prediction date."
        },
        {
            "feature": "feature_trailing_market_5d",
            "earliest_source_offset": -5,
            "latest_source_offset": -1,
            "available_offset": 0,
            "description": "Trailing market return known at prediction date."
        },
        {
            "feature": "feature_same_day_cross_sectional_rank",
            "earliest_source_offset": 0,
            "latest_source_offset": 0,
            "available_offset": 0,
            "description": "Same-day cross-sectional rank; safe only for after-close decisions."
        },
    ])

    feature_cols = feature_catalog["feature"].tolist()

    out = out.dropna(subset=feature_cols + ["label_forward_log_return"]).copy()

    return out, feature_catalog


safe_panel, safe_feature_catalog = build_safe_features(labelled_panel)

safe_panel.head()

In [ ]:
safe_feature_catalog

## 7. Deliberately leaky features

We now create features that should never be allowed into a real model.

Examples:

1. using the future label as a feature;
2. using next-day return;
3. using a centered rolling mean;
4. using future realised volatility;
5. using a future cross-sectional rank.

These are intentionally bad. The unit tests later should catch them.

In [ ]:
def build_leaky_features(labelled: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build deliberately leaky features and a feature catalog.
    """
    out = labelled.sort_values(["symbol", "timestamp"]).copy()
    grouped = out.groupby("symbol")

    out["feature_lag1_return"] = grouped["return"].shift(1)

    # Leaky: the label itself.
    out["feature_future_label_copy"] = out["label_forward_log_return"]

    # Leaky: next day's return.
    out["feature_next_day_return"] = grouped["return"].shift(-1)

    # Leaky: centered rolling window includes future observations.
    out["feature_centered_5d_return"] = (
        grouped["return"]
        .rolling(window=5, min_periods=5, center=True)
        .sum()
        .reset_index(level=0, drop=True)
    )

    # Leaky: future realised volatility over the label horizon.
    out["feature_future_5d_vol"] = (
        grouped["return"]
        .rolling(window=5, min_periods=5)
        .std()
        .reset_index(level=0, drop=True)
        .shift(-5)
    )

    # Leaky: future rank of the forward label.
    out["feature_future_label_rank"] = (
        out.groupby("timestamp")["label_forward_log_return"]
        .rank(pct=True)
    )

    feature_catalog = pd.DataFrame([
        {
            "feature": "feature_lag1_return",
            "earliest_source_offset": -1,
            "latest_source_offset": -1,
            "available_offset": 0,
            "description": "Safe lagged return."
        },
        {
            "feature": "feature_future_label_copy",
            "earliest_source_offset": 1,
            "latest_source_offset": config.label_horizon,
            "available_offset": config.label_horizon,
            "description": "Leaky copy of the future label."
        },
        {
            "feature": "feature_next_day_return",
            "earliest_source_offset": 1,
            "latest_source_offset": 1,
            "available_offset": 1,
            "description": "Leaky next-day return."
        },
        {
            "feature": "feature_centered_5d_return",
            "earliest_source_offset": -2,
            "latest_source_offset": 2,
            "available_offset": 2,
            "description": "Centered rolling return includes future observations."
        },
        {
            "feature": "feature_future_5d_vol",
            "earliest_source_offset": 1,
            "latest_source_offset": 5,
            "available_offset": 5,
            "description": "Future realised volatility over label horizon."
        },
        {
            "feature": "feature_future_label_rank",
            "earliest_source_offset": 1,
            "latest_source_offset": config.label_horizon,
            "available_offset": config.label_horizon,
            "description": "Cross-sectional rank of the future label."
        },
    ])

    feature_cols = feature_catalog["feature"].tolist()
    out = out.dropna(subset=feature_cols + ["label_forward_log_return"]).copy()

    return out, feature_catalog


leaky_panel, leaky_feature_catalog = build_leaky_features(labelled_panel)

leaky_feature_catalog

## 8. Basic predictive diagnostic

Before writing tests, we show why leakage is dangerous.

A simple information coefficient is:

$$
IC = \operatorname{corr}(x_t, y_t)
$$
where $x_t$ is a feature and $y_t$ is the future return label.

A leaky feature can show an unrealistically high IC.

In [ ]:
def information_coefficient_table(
    panel: pd.DataFrame,
    feature_catalog: pd.DataFrame,
    label_col: str = "label_forward_log_return"
) -> pd.DataFrame:
    """
    Compute feature-label correlations.
    """
    rows = []

    for feature in feature_catalog["feature"]:
        valid = panel[[feature, label_col]].replace([np.inf, -np.inf], np.nan).dropna()

        if len(valid) < 10:
            ic = np.nan
        else:
            ic = valid[feature].corr(valid[label_col])

        rows.append({
            "feature": feature,
            "information_coefficient": ic,
            "abs_ic": abs(ic)
        })

    return pd.DataFrame(rows).sort_values("abs_ic", ascending=False)


safe_ic = information_coefficient_table(safe_panel, safe_feature_catalog)
leaky_ic = information_coefficient_table(leaky_panel, leaky_feature_catalog)

safe_ic

In [ ]:
leaky_ic

In [ ]:
plt.figure(figsize=(10, 6))
plot_data = leaky_ic.sort_values("abs_ic", ascending=True)
plt.barh(plot_data["feature"], plot_data["abs_ic"])
plt.title("Leaky Feature Absolute Information Coefficients")
plt.xlabel("Absolute IC")
plt.ylabel("Feature")
plt.show()

## 9. Test 1 — Feature source-time leakage

A feature is safe only if its latest source observation is no later than the prediction timestamp.

Using offset notation:

- $-1$: yesterday;
- $0$: prediction timestamp;
- $+1$: tomorrow.

A feature has source-time leakage if:

$$
\text{latest source offset} > 0
$$
This metadata-based test is simple and powerful.

It requires the feature engineering code to maintain a feature catalog.

In [ ]:
class LeakageTestError(AssertionError):
    """
    Custom assertion error for leakage tests.
    """


def assert_no_future_source_offsets(feature_catalog: pd.DataFrame) -> None:
    """
    Assert that all features use source data available no later than prediction time.
    """
    offenders = feature_catalog[feature_catalog["latest_source_offset"] > 0].copy()

    if not offenders.empty:
        raise LeakageTestError(
            "Future-source leakage detected in features: "
            + ", ".join(offenders["feature"].tolist())
        )


def run_expected_failure(test_func, *args, **kwargs) -> str:
    """
    Run a test that is expected to fail and return the error message.
    """
    try:
        test_func(*args, **kwargs)
    except AssertionError as exc:
        return str(exc)

    raise AssertionError("Expected the test to fail, but it passed.")

In [ ]:
# Safe catalog should pass.
assert_no_future_source_offsets(safe_feature_catalog)
print("Safe feature catalog passed source-time leakage test.")

# Leaky catalog should fail.
expected_error = run_expected_failure(
    assert_no_future_source_offsets,
    leaky_feature_catalog
)

expected_error

## 10. Test 2 — Feature availability before label start

A feature should be available before the label interval begins.

If a feature becomes available only after the label starts, it is using information from the prediction horizon.

We test:

$$
\text{available offset} \leq 0
$$
for a same-day after-close prediction convention.

For intraday systems, this test would need more precise timestamps.

In [ ]:
def assert_features_available_before_label_start(feature_catalog: pd.DataFrame) -> None:
    """
    Assert that all features are available by prediction time.
    """
    offenders = feature_catalog[feature_catalog["available_offset"] > 0].copy()

    if not offenders.empty:
        raise LeakageTestError(
            "Feature availability leakage detected: "
            + ", ".join(offenders["feature"].tolist())
        )


assert_features_available_before_label_start(safe_feature_catalog)
print("Safe feature catalog passed feature-availability test.")

expected_error = run_expected_failure(
    assert_features_available_before_label_start,
    leaky_feature_catalog
)

expected_error

## 11. Test 3 — Target-copy leakage

Some leakage can be detected statistically.

A feature that is almost perfectly correlated with the label is suspicious.

This test is not a proof, because a real feature could be highly predictive in a toy dataset. But in finance, a feature-label correlation near 1 is almost always a red flag.

We flag:

$$
|\operatorname{corr}(x,y)| > 0.95
$$

In [ ]:
def assert_no_suspicious_target_correlation(
    panel: pd.DataFrame,
    feature_catalog: pd.DataFrame,
    label_col: str = "label_forward_log_return",
    threshold: float = 0.95
) -> None:
    """
    Assert no feature is suspiciously correlated with the label.
    """
    ic_table = information_coefficient_table(panel, feature_catalog, label_col=label_col)
    offenders = ic_table[ic_table["abs_ic"] > threshold].copy()

    if not offenders.empty:
        raise LeakageTestError(
            "Suspicious target correlation detected: "
            + ", ".join(
                f"{row.feature} IC={row.information_coefficient:.4f}"
                for row in offenders.itertuples()
            )
        )


assert_no_suspicious_target_correlation(safe_panel, safe_feature_catalog, threshold=0.95)
print("Safe feature panel passed suspicious-correlation test.")

expected_error = run_expected_failure(
    assert_no_suspicious_target_correlation,
    leaky_panel,
    leaky_feature_catalog,
    threshold=0.95
)

expected_error

## 12. Time-ordered train/test split

For time-series research, train/test splits should preserve temporal order.

A basic split is:

$$
\text{train dates} < \text{test dates}
$$
Random shuffling is usually inappropriate because it lets the model train on future regimes and test on the past.

Scikit-learn's `TimeSeriesSplit` exists for time-ordered cross-validation, and its documentation explicitly notes that other cross-validation methods are inappropriate for time-ordered data because they can train on future data and evaluate on past data.

In [ ]:
def make_time_ordered_split(
    panel: pd.DataFrame,
    train_fraction: float
) -> tuple[pd.Index, pd.Index]:
    """
    Create a time-ordered train/test split by timestamp.
    """
    unique_dates = pd.DatetimeIndex(sorted(panel["timestamp"].unique()))
    split_idx = int(len(unique_dates) * train_fraction)

    train_dates = set(unique_dates[:split_idx])
    test_dates = set(unique_dates[split_idx:])

    train_idx = panel.index[panel["timestamp"].isin(train_dates)]
    test_idx = panel.index[panel["timestamp"].isin(test_dates)]

    return train_idx, test_idx


def make_random_split(
    panel: pd.DataFrame,
    train_fraction: float,
    seed: int = 42
) -> tuple[pd.Index, pd.Index]:
    """
    Create a deliberately unsafe random train/test split.
    """
    local_rng = np.random.default_rng(seed)
    indices = panel.index.to_numpy()
    local_rng.shuffle(indices)

    split = int(len(indices) * train_fraction)

    return pd.Index(indices[:split]), pd.Index(indices[split:])

In [ ]:
train_idx, test_idx = make_time_ordered_split(safe_panel, config.train_fraction)
random_train_idx, random_test_idx = make_random_split(safe_panel, config.train_fraction)

safe_panel.loc[train_idx, "timestamp"].max(), safe_panel.loc[test_idx, "timestamp"].min()

## 13. Test 4 — Split respects time order

A split is time-ordered if:

$$
\max(t_{\text{train}}) < \min(t_{\text{test}})
$$
This test should pass for the time-ordered split and fail for the random split.

In [ ]:
def assert_train_before_test(
    panel: pd.DataFrame,
    train_idx: pd.Index,
    test_idx: pd.Index
) -> None:
    """
    Assert all training timestamps occur before all test timestamps.
    """
    train_max = panel.loc[train_idx, "timestamp"].max()
    test_min = panel.loc[test_idx, "timestamp"].min()

    if not train_max < test_min:
        raise LeakageTestError(
            f"Train/test time order violated: train_max={train_max}, test_min={test_min}"
        )


assert_train_before_test(safe_panel, train_idx, test_idx)
print("Time-ordered split passed.")

expected_error = run_expected_failure(
    assert_train_before_test,
    safe_panel,
    random_train_idx,
    random_test_idx
)

expected_error

## 14. Preprocessing leakage

Preprocessing leakage happens when transformations are fitted on the full dataset before splitting.

Example:

```python
global_mean = X.mean()
global_std = X.std()
X_scaled = (X - global_mean) / global_std
```

If `global_mean` and `global_std` use test data, the training data has learned something about the test distribution.

Correct workflow:

1. split into train and test;
2. fit scaler on train only;
3. transform train and test using train-fitted parameters.

Scikit-learn's common-pitfalls documentation gives the same core advice: split data into train and test before preprocessing, and avoid using test data when fitting preprocessing steps.

In [ ]:
def fit_standardizer(
    panel: pd.DataFrame,
    feature_cols: list[str],
    fit_idx: pd.Index,
    fitted_until: pd.Timestamp
) -> dict:
    """
    Fit a simple standardizer on selected rows.
    """
    x = panel.loc[fit_idx, feature_cols]
    mean = x.mean()
    std = x.std(ddof=0).replace(0, 1.0)

    return {
        "feature_cols": feature_cols,
        "mean": mean,
        "std": std,
        "fitted_until": fitted_until
    }


def transform_with_standardizer(panel: pd.DataFrame, standardizer: dict) -> pd.DataFrame:
    """
    Apply a fitted standardizer.
    """
    out = panel.copy()
    feature_cols = standardizer["feature_cols"]

    for col in feature_cols:
        out[f"scaled_{col}"] = (out[col] - standardizer["mean"][col]) / standardizer["std"][col]

    return out


safe_feature_cols = safe_feature_catalog["feature"].tolist()

train_end = safe_panel.loc[train_idx, "timestamp"].max()
full_end = safe_panel["timestamp"].max()

train_only_standardizer = fit_standardizer(
    safe_panel,
    safe_feature_cols,
    train_idx,
    fitted_until=train_end
)

global_leaky_standardizer = fit_standardizer(
    safe_panel,
    safe_feature_cols,
    safe_panel.index,
    fitted_until=full_end
)

train_only_standardizer["fitted_until"], global_leaky_standardizer["fitted_until"]

In [ ]:
def assert_preprocessor_fit_on_train_only(
    preprocessor_metadata: dict,
    train_end_time: pd.Timestamp
) -> None:
    """
    Assert preprocessing was fitted no later than the training end timestamp.
    """
    fitted_until = preprocessor_metadata["fitted_until"]

    if fitted_until > train_end_time:
        raise LeakageTestError(
            f"Preprocessor was fitted beyond train end: fitted_until={fitted_until}, "
            f"train_end={train_end_time}"
        )


assert_preprocessor_fit_on_train_only(train_only_standardizer, train_end)
print("Train-only preprocessing metadata passed.")

expected_error = run_expected_failure(
    assert_preprocessor_fit_on_train_only,
    global_leaky_standardizer,
    train_end
)

expected_error

## 15. Overlapping-label leakage

Financial labels often span intervals.

For example, if the label is a 5-day forward return, then the sample at date $t$ uses returns from:

$$
(t, t+5]
$$
If a training label interval overlaps with a test label interval, information can leak through overlapping outcomes.

A simple overlap condition is:

$$
\text{train label start} \leq \text{test label end}
\quad \text{and} \quad
\text{train label end} \geq \text{test label start}
$$
Purged cross-validation removes training samples whose label intervals overlap with the test label interval.

An embargo then removes samples shortly after the test period to reduce leakage from serial dependence or delayed information.

In [ ]:
def label_intervals_overlap(
    train_start: pd.Series,
    train_end: pd.Series,
    test_start_min: pd.Timestamp,
    test_end_max: pd.Timestamp
) -> pd.Series:
    """
    Return mask of training rows whose label interval overlaps the test label window.
    """
    return (train_start <= test_end_max) & (train_end >= test_start_min)


def purge_and_embargo_split(
    panel: pd.DataFrame,
    train_idx: pd.Index,
    test_idx: pd.Index,
    embargo_days: int
) -> tuple[pd.Index, pd.Index, pd.DataFrame]:
    """
    Remove training samples whose label intervals overlap the test label window,
    then apply an embargo after the test window.
    """
    test_label_start_min = panel.loc[test_idx, "label_start_time"].min()
    test_label_end_max = panel.loc[test_idx, "label_end_time"].max()

    train_rows = panel.loc[train_idx].copy()

    overlap_mask = label_intervals_overlap(
        train_rows["label_start_time"],
        train_rows["label_end_time"],
        test_label_start_min,
        test_label_end_max
    )

    embargo_end = test_label_end_max + pd.Timedelta(days=embargo_days)

    embargo_mask = (
        (train_rows["timestamp"] > test_label_end_max)
        & (train_rows["timestamp"] <= embargo_end)
    )

    keep_mask = ~(overlap_mask | embargo_mask)

    purged_train_idx = train_rows.index[keep_mask]

    audit = pd.DataFrame({
        "rule": ["label_overlap_purge", "embargo"],
        "removed_rows": [int(overlap_mask.sum()), int(embargo_mask.sum())],
        "test_label_start_min": [test_label_start_min, test_label_start_min],
        "test_label_end_max": [test_label_end_max, test_label_end_max],
        "embargo_end": [embargo_end, embargo_end]
    })

    return pd.Index(purged_train_idx), test_idx, audit

In [ ]:
purged_train_idx, purged_test_idx, purge_audit = purge_and_embargo_split(
    safe_panel,
    train_idx,
    test_idx,
    embargo_days=config.embargo_days
)

purge_audit

## 16. Test 5 — No train/test label interval overlap

The purged split should pass the overlap test.

A deliberately bad split with mixed dates should fail.

In [ ]:
def assert_no_label_overlap_between_train_and_test(
    panel: pd.DataFrame,
    train_idx: pd.Index,
    test_idx: pd.Index
) -> None:
    """
    Assert no training label interval overlaps the overall test label interval.
    """
    test_label_start_min = panel.loc[test_idx, "label_start_time"].min()
    test_label_end_max = panel.loc[test_idx, "label_end_time"].max()

    train_rows = panel.loc[train_idx]

    overlap_mask = label_intervals_overlap(
        train_rows["label_start_time"],
        train_rows["label_end_time"],
        test_label_start_min,
        test_label_end_max
    )

    if overlap_mask.any():
        raise LeakageTestError(
            f"Label overlap leakage detected: {int(overlap_mask.sum())} training rows overlap test label window."
        )


assert_no_label_overlap_between_train_and_test(
    safe_panel,
    purged_train_idx,
    purged_test_idx
)

print("Purged split passed label-overlap test.")

# Random split should usually fail.
expected_error = run_expected_failure(
    assert_no_label_overlap_between_train_and_test,
    safe_panel,
    random_train_idx,
    random_test_idx
)

expected_error

## 17. Availability-time leakage example

Some data has an event timestamp and a release timestamp.

Example:

- earnings quarter end;
- earnings release date;
- macro period end;
- macro release date;
- corporate action announcement time;
- index membership announcement.

Using event-time data before release time creates look-ahead bias.

Below we simulate a feature with:

- `event_time`;
- `released_at`;
- `feature_timestamp`.

The test checks:

$$
\text{released at} \leq \text{feature timestamp}
$$

In [ ]:
def simulate_availability_time_data(panel: pd.DataFrame, seed: int = 42) -> pd.DataFrame:
    """
    Simulate a fundamental-like feature with event time and release time.
    """
    local_rng = np.random.default_rng(seed)

    unique_dates = pd.DatetimeIndex(sorted(panel["timestamp"].unique()))
    symbols = panel["symbol"].unique()

    rows = []

    for symbol in symbols[:10]:
        report_dates = unique_dates[::63]

        for event_time in report_dates:
            lag_days = int(local_rng.integers(5, 25))
            released_at = event_time + pd.Timedelta(days=lag_days)

            feature_value = local_rng.normal()

            rows.append({
                "symbol": symbol,
                "event_time": event_time,
                "released_at": released_at,
                "fundamental_value": feature_value
            })

    fundamentals = pd.DataFrame(rows)

    return fundamentals


fundamentals = simulate_availability_time_data(safe_panel, seed=42)

fundamentals.head()

In [ ]:
def make_fundamental_join_examples(
    panel: pd.DataFrame,
    fundamentals: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Create safe and leaky joins for availability-time testing.
    """
    base = panel[["timestamp", "symbol"]].drop_duplicates().copy()

    # Leaky: join by event_time, as if the value were known at quarter end.
    leaky_join = base.merge(
        fundamentals,
        left_on=["symbol", "timestamp"],
        right_on=["symbol", "event_time"],
        how="left"
    )

    leaky_join["feature_timestamp"] = leaky_join["timestamp"]

    # Safe: for each feature timestamp, use most recent released value.
    safe_rows = []

    for symbol, group in base.groupby("symbol"):
        f = fundamentals[fundamentals["symbol"] == symbol].sort_values("released_at")
        g = group.sort_values("timestamp")

        if f.empty:
            g = g.copy()
            g["event_time"] = pd.NaT
            g["released_at"] = pd.NaT
            g["fundamental_value"] = np.nan
            safe_rows.append(g)
            continue

        merged = pd.merge_asof(
            g.sort_values("timestamp"),
            f.sort_values("released_at"),
            left_on="timestamp",
            right_on="released_at",
            by="symbol",
            direction="backward"
        )

        merged["feature_timestamp"] = merged["timestamp"]
        safe_rows.append(merged)

    safe_join = pd.concat(safe_rows, ignore_index=True)

    return safe_join, leaky_join


safe_fundamental_join, leaky_fundamental_join = make_fundamental_join_examples(
    safe_panel,
    fundamentals
)

safe_fundamental_join.head()

In [ ]:
def assert_released_before_feature_timestamp(joined: pd.DataFrame) -> None:
    """
    Assert all non-null feature values were released before the feature timestamp.
    """
    used = joined[joined["fundamental_value"].notna()].copy()

    offenders = used[used["released_at"] > used["feature_timestamp"]]

    if not offenders.empty:
        raise LeakageTestError(
            f"Availability-time leakage detected: {len(offenders)} rows use data before release."
        )


assert_released_before_feature_timestamp(safe_fundamental_join)
print("Safe fundamental join passed availability-time test.")

expected_error = run_expected_failure(
    assert_released_before_feature_timestamp,
    leaky_fundamental_join
)

expected_error

## 18. Unit-test runner

We now collect tests into a simple runner.

In a real repository, these functions could be moved into:

```text
tests/test_data_leakage.py
```

and executed with:

```bash
pytest
```

For the notebook, the runner gives a compact pass/fail report.

In [ ]:
@dataclass
class TestResult:
    """
    Simple unit-test result container.
    """
    test_name: str
    expected: str
    outcome: str
    message: str


def run_test_case(
    test_name: str,
    func,
    args: tuple,
    expected: str
) -> TestResult:
    """
    Run one test and record whether it passed or failed.
    """
    try:
        func(*args)
        actual = "pass"
        message = "OK"
    except AssertionError as exc:
        actual = "fail"
        message = str(exc)

    if actual == expected:
        outcome = "as_expected"
    else:
        outcome = "unexpected"

    return TestResult(
        test_name=test_name,
        expected=expected,
        outcome=outcome,
        message=message
    )


test_cases = [
    (
        "safe_features_no_future_source_offsets",
        assert_no_future_source_offsets,
        (safe_feature_catalog,),
        "pass"
    ),
    (
        "leaky_features_future_source_offsets",
        assert_no_future_source_offsets,
        (leaky_feature_catalog,),
        "fail"
    ),
    (
        "safe_features_available_before_label_start",
        assert_features_available_before_label_start,
        (safe_feature_catalog,),
        "pass"
    ),
    (
        "leaky_features_available_before_label_start",
        assert_features_available_before_label_start,
        (leaky_feature_catalog,),
        "fail"
    ),
    (
        "safe_panel_no_target_copy_correlation",
        assert_no_suspicious_target_correlation,
        (safe_panel, safe_feature_catalog),
        "pass"
    ),
    (
        "leaky_panel_target_copy_correlation",
        assert_no_suspicious_target_correlation,
        (leaky_panel, leaky_feature_catalog),
        "fail"
    ),
    (
        "time_ordered_split_train_before_test",
        assert_train_before_test,
        (safe_panel, train_idx, test_idx),
        "pass"
    ),
    (
        "random_split_train_before_test",
        assert_train_before_test,
        (safe_panel, random_train_idx, random_test_idx),
        "fail"
    ),
    (
        "train_only_preprocessor",
        assert_preprocessor_fit_on_train_only,
        (train_only_standardizer, train_end),
        "pass"
    ),
    (
        "global_leaky_preprocessor",
        assert_preprocessor_fit_on_train_only,
        (global_leaky_standardizer, train_end),
        "fail"
    ),
    (
        "purged_split_no_label_overlap",
        assert_no_label_overlap_between_train_and_test,
        (safe_panel, purged_train_idx, purged_test_idx),
        "pass"
    ),
    (
        "random_split_label_overlap",
        assert_no_label_overlap_between_train_and_test,
        (safe_panel, random_train_idx, random_test_idx),
        "fail"
    ),
    (
        "safe_fundamental_join_release_time",
        assert_released_before_feature_timestamp,
        (safe_fundamental_join,),
        "pass"
    ),
    (
        "leaky_fundamental_join_release_time",
        assert_released_before_feature_timestamp,
        (leaky_fundamental_join,),
        "fail"
    ),
]

results = [
    run_test_case(name, func, args, expected)
    for name, func, args, expected in test_cases
]

test_results_df = pd.DataFrame([asdict(result) for result in results])

test_results_df

In [ ]:
plt.figure(figsize=(8, 5))
test_results_df["outcome"].value_counts().plot(kind="bar")
plt.title("Leakage Test Outcomes")
plt.xlabel("Outcome")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

## 19. Leakage scorecard

A practical pipeline can convert test results into a scorecard.

A model should not be evaluated if any leakage-critical test has an unexpected result.

This is a governance layer, not just a notebook convenience.

In [ ]:
def build_leakage_scorecard(test_results: pd.DataFrame) -> dict:
    """
    Build a compact leakage scorecard.
    """
    total = len(test_results)
    unexpected = int((test_results["outcome"] == "unexpected").sum())
    as_expected = int((test_results["outcome"] == "as_expected").sum())

    return {
        "schema_version": "leakage_test_scorecard_v1",
        "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
        "total_tests": total,
        "as_expected": as_expected,
        "unexpected": unexpected,
        "status": "pass" if unexpected == 0 else "fail"
    }


leakage_scorecard = build_leakage_scorecard(test_results_df)

leakage_scorecard

## 20. Saving test artifacts

The notebook saves:

1. safe feature catalog;
2. leaky feature catalog;
3. test results;
4. leakage scorecard;
5. purge audit report;
6. information coefficient diagnostics.

These can be used as templates for future notebooks.

In [ ]:
output_dir = Path("data/processed/data_leakage_tests_v1")
output_dir.mkdir(parents=True, exist_ok=True)

safe_catalog_path = output_dir / "safe_feature_catalog.csv"
leaky_catalog_path = output_dir / "leaky_feature_catalog.csv"
test_results_path = output_dir / "leakage_test_results.csv"
scorecard_path = output_dir / "leakage_scorecard.json"
purge_audit_path = output_dir / "purge_embargo_audit.csv"
safe_ic_path = output_dir / "safe_information_coefficients.csv"
leaky_ic_path = output_dir / "leaky_information_coefficients.csv"

safe_feature_catalog.to_csv(safe_catalog_path, index=False)
leaky_feature_catalog.to_csv(leaky_catalog_path, index=False)
test_results_df.to_csv(test_results_path, index=False)
purge_audit.to_csv(purge_audit_path, index=False)
safe_ic.to_csv(safe_ic_path, index=False)
leaky_ic.to_csv(leaky_ic_path, index=False)

scorecard_path.write_text(json.dumps(leakage_scorecard, indent=2, default=str))

safe_catalog_path, leaky_catalog_path, test_results_path, scorecard_path

## 21. Practical leakage checklist

Before trusting a financial ML or signal research notebook, ask:

1. **Feature source time**  
   Does every feature use only data available at or before prediction time?

2. **Feature availability time**  
   Was the feature actually released before the prediction?

3. **Label interval**  
   What future interval does the label cover?

4. **Train/test order**  
   Are all training timestamps before test timestamps?

5. **Preprocessing**  
   Were scalers, imputers, PCA, feature selectors, and encoders fitted only on training data?

6. **Overlapping labels**  
   Do training labels overlap test labels?

7. **Embargo**  
   Is there a gap after test windows where necessary?

8. **Universe construction**  
   Was the universe known at the time, or reconstructed using future survivors?

9. **Hyperparameter search**  
   Was test data used repeatedly during model selection?

10. **Suspicious performance**  
   Are IC, Sharpe, hit rate, or accuracy unrealistically high?

11. **Metadata**  
   Does each feature have source-time and availability-time metadata?

12. **Unit tests**  
   Do leakage tests run automatically in CI or before model evaluation?

## 22. Limitations

This notebook deliberately simplifies leakage detection.

### 22.1 Metadata must be maintained

The source-offset tests work only if feature engineering code records metadata.

If the metadata is wrong, the test may pass incorrectly.

### 22.2 Statistical tests cannot prove absence of leakage

A suspiciously high correlation can flag target leakage, but a normal-looking correlation does not prove the pipeline is safe.

Metadata and timestamp tests are more reliable.

### 22.3 Real release times are more complex

Fundamental, macro, and corporate-action data may have:

- event dates;
- announcement times;
- vendor receipt times;
- revision times;
- timezone issues;
- exchange-specific calendars.

The notebook uses daily timestamps for simplicity.

### 22.4 Purging rules depend on label design

The correct purging and embargo logic depends on:

- label horizon;
- overlapping outcomes;
- event-based labels;
- barrier labels;
- holding periods;
- serial dependence;
- execution delay.

### 22.5 Random split is not always wrong, but usually suspicious

For strictly cross-sectional problems with no temporal leakage and independent samples, random splits may be defensible.

For most financial time-series prediction tasks, preserving time order is safer.

### 22.6 Passing leakage tests does not guarantee a good model

These tests only detect certain classes of invalid research design.

A model can be leakage-free and still be overfit, unstable, costly to trade, or economically meaningless.

## 23. Research frontier and current directions

Leakage control is now part of serious model validation and research governance.

### 23.1 Purged and embargoed cross-validation

Financial labels often overlap in time.

Purged cross-validation removes training samples whose label intervals overlap the test period, while embargoing removes samples immediately after the test period.

A future notebook could implement full combinatorial purged cross-validation and compare it with ordinary K-fold and walk-forward splits.

### 23.2 Leakage-aware feature stores

Modern ML platforms increasingly use feature stores to track:

- event time;
- processing time;
- availability time;
- training dataset version;
- point-in-time correctness.

A future notebook could design a mini feature store for market, fundamental, and macro features.

### 23.3 Bitemporal data validation

Bitemporal data tracks both when an event happened and when the data became known.

This is crucial for fundamentals and macroeconomic data, where revisions can create look-ahead bias.

A future notebook could simulate GDP revisions and test how using final revised data inflates performance.

### 23.4 Automated research linting

Financial research notebooks could be linted for common leakage patterns:

- `.shift(-1)` in feature columns;
- centered rolling windows;
- global `fit_transform`;
- random train/test splits;
- post-hoc universe filtering;
- using adjusted data incorrectly;
- joining on event date instead of release date.

A future project could build a notebook linter that flags suspicious patterns before code review.

### 23.5 Backtest overfitting and multiple testing

Leakage is one source of fake performance. Multiple testing is another.

Even leakage-free pipelines can overfit if thousands of strategies are tried and only the best one is reported.

A future notebook could connect leakage tests to deflated Sharpe ratio, probability of backtest overfitting, and White's Reality Check.

### 23.6 CI/CD for quant research

A mature quant repository should run tests automatically.

For example:

```bash
pytest tests/test_data_leakage.py
pytest tests/test_no_lookahead.py
pytest tests/test_feature_availability.py
```

A future notebook could convert the tests in this notebook into a proper package-level test suite.

## 24. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_14_information_coefficient_and_feature_decay**  
   Measuring signal quality after leakage tests pass.

2. **05_04_walk_forward_optimization**  
   Evaluating models using temporally ordered out-of-sample windows.

3. **05_06_performance_haircut_and_deflated_sharpe**  
   Adjusting performance for selection bias and multiple testing.

4. **05_07_purged_kfold_and_embargo_cv**  
   Implementing full purged cross-validation for financial labels.

5. **05_08_probability_of_backtest_overfitting**  
   Quantifying how strategy mining creates false discoveries.

6. **05_09_white_reality_check_bootstrap**  
   Testing whether the best strategy beats data-mined alternatives.

7. **07_01_multi_asset_cta_research_pipeline**  
   Running leakage checks as a gate before any end-to-end backtest.

## 25. Summary

This notebook built a leakage-detection test suite for financial time-series research.

It demonstrated:

1. safe feature engineering;
2. deliberately leaky features;
3. feature source-time metadata;
4. feature availability-time metadata;
5. target-copy leakage detection;
6. time-ordered train/test splitting;
7. preprocessing leakage checks;
8. purging and embargoing for overlapping labels;
9. release-time checks for fundamental data;
10. a reusable test runner and leakage scorecard.

The key computational takeaway is:

> Leakage tests should be automated. They should not rely on a researcher remembering every possible way future information can enter the model.

The key financial takeaway is:

> Most impressive backtests are not impressive until the pipeline proves that the model could only use information available at the time.

## 26. Further reading

### Time-series validation and leakage

- scikit-learn documentation on `TimeSeriesSplit`.
- scikit-learn documentation on common pitfalls and data leakage.
- López de Prado, M. *Advances in Financial Machine Learning.*
- Literature on purged and embargoed cross-validation.
- Literature on backtest overfitting and multiple testing.

### Point-in-time data

- Feature-store documentation on point-in-time joins.
- Bitemporal data modelling.
- Macroeconomic data revisions and release calendars.
- Corporate-action and index-membership availability timestamps.

### Testing and governance

- pytest documentation.
- CI/CD for scientific Python.
- Reproducible ML pipelines.
- Model risk management.
- Quant research code review and linting.